[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [scikit-learn, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)

# Classification Metrics &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def station_network(n=1200, seed=2025):
    """A made-up network of weather stations in Norway, one row each, with three things to predict."""
    rng = np.random.default_rng(seed)
    latitude = rng.uniform(58.0, 71.0, n)
    longitude = 5 + 1.5 * (latitude - 58) + rng.uniform(0, 5, n)
    coast_km = np.minimum(rng.exponential(25, n), 250)
    elevation = np.clip(rng.gamma(1.5, 150, n) + 3 * coast_km, 0, 1800)
    terrain = np.where(coast_km < 3, "coastal",
              np.where(elevation > 700, "mountain",
              np.where((coast_km < 30) & (rng.uniform(size=n) < 0.6), "fjord", "inland")))
    owner = rng.choice(["met", "road", "airport", "private"], n, p=[0.4, 0.3, 0.1, 0.2])
    installed = rng.integers(1950, 2025, n)
    shift = pd.Series(terrain).map({"coastal": 0.8, "fjord": 0.3, "inland": 0.0, "mountain": -0.5}).to_numpy()
    jan_mean = (1.5 - 0.55 * (latitude - 58) - 0.006 * elevation - 4.0 * (1 - np.exp(-coast_km / 40))
                + shift + rng.normal(0, 0.9, n))
    may_mean = jan_mean + 8.5 + 0.004 * coast_km + rng.normal(0, 1.2, n)
    june_min = 8.5 - 0.35 * (latitude - 58) - 0.0075 * elevation - 0.02 * coast_km + rng.normal(0, 1.5, n)
    table = pd.DataFrame({
        "station": [f"ST{i:04d}" for i in range(1, n + 1)],
        "latitude": latitude.round(2), "longitude": longitude.round(2),
        "elevation_m": elevation.round(0).astype(int), "coast_km": coast_km.round(1),
        "terrain": terrain, "owner": owner, "installed": installed,
        "jan_mean": jan_mean.round(1),
        "snow_on_may_1": may_mean < 2.5,
        "june_frost": june_min < -0.8,
    })
    return table


network = station_network()
FEATURES = ["latitude", "longitude", "elevation_m", "coast_km", "installed"]

print("scikit-learn", sklearn.__version__, "|", len(network), "stations")

scikit-learn 1.6.1 | 1200 stations


**1.** Print the confusion matrix for the threshold of 0.2, as a labeled DataFrame like the worked example's.

In [2]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
snow_chance = model.predict_proba(X_test)[:, 1]

at_two_tenths = snow_chance > 0.2

pd.DataFrame(confusion_matrix(y_test, at_two_tenths), index=["clear", "snow"],
             columns=["predicted clear", "predicted snow"])

,predicted clear,predicted snow
clear,139,45
snow,9,107


The first five lines rebuild the split and the model as the worked examples did, since this notebook runs on its own. At 0.2 the model flags 152 stations, 107 of them snowy and 45 false alarms, and misses 9 snowy stations.

**2.** Print `classification_report` for the predictions at 0.2, and say which of the snow row's three numbers rose and which fell compared with 0.5.

In [3]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
snow_chance = model.predict_proba(X_test)[:, 1]

print("at 0.5:")
print(classification_report(y_test, snow_chance > 0.5, digits=3))
print("at 0.2:")
print(classification_report(y_test, snow_chance > 0.2, digits=3))

at 0.5:
              precision    recall  f1-score   support

       False      0.851     0.897     0.873       184
        True      0.821     0.750     0.784       116

    accuracy                          0.840       300
   macro avg      0.836     0.823     0.828       300
weighted avg      0.839     0.840     0.839       300

at 0.2:
              precision    recall  f1-score   support

       False      0.939     0.755     0.837       184
        True      0.704     0.922     0.799       116

    accuracy                          0.820       300
   macro avg      0.822     0.839     0.818       300
weighted avg      0.848     0.820     0.822       300



In the `True` row, recall rose from 0.75 to 0.922 and precision fell from 0.821 to 0.704. F1 rose slightly, from 0.784 to 0.799, so on F1 alone the two thresholds are close.

**3.** Price a false alarm at 1 and a miss at 10 instead, and print the threshold from the table with the lowest cost.

In [4]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
snow_chance = model.predict_proba(X_test)[:, 1]

train_chance = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression()), X_train, y_train,
                                 cv=StratifiedKFold(5, shuffle=True, random_state=0), method="predict_proba")[:, 1]
costs = {}
for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    flagged = train_chance > threshold
    false_alarms = int((flagged & ~y_train).sum())
    missed = int((~flagged & y_train).sum())
    costs[threshold] = false_alarms * 1 + missed * 10

print(costs)
print("cheapest:", min(costs, key=costs.get))

{0.1: 316, 0.2: 392, 0.3: 525, 0.4: 594, 0.5: 734, 0.6: 1049, 0.7: 1375, 0.8: 1671, 0.9: 2237}
cheapest: 0.1


The table is built from the training stations' out-of-fold probabilities, as in the worked example. With a miss ten times as costly as a false alarm, the cheapest threshold is 0.1, the lowest one in the table, so a wider table would be the next thing to try.

**4.** Fit a `DecisionTreeClassifier(max_depth=4, random_state=0)` on the same training set, and print its precision and recall on the test set at the default threshold.

In [5]:
X = network[FEATURES]
y = network["snow_on_may_1"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
model = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
snow_chance = model.predict_proba(X_test)[:, 1]

from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)
tree_predicted = tree.predict(X_test)

print("precision:", round(precision_score(y_test, tree_predicted), 3))
print("recall:   ", round(recall_score(y_test, tree_predicted), 3))

precision: 0.84
recall:    0.681


The metrics do not care which model made the predictions. The tree's precision, 0.84, is higher than the logistic regression's 0.821, and its recall, 0.681, is lower than 0.75. `DecisionTreeClassifier` is imported in this cell because the notebook's Setup has no tree.

---

&#8592; **Back to:** [Classification Metrics](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/scikit-learn-deep-dive/08-classification-metrics.ipynb)  &nbsp;&middot;&nbsp;  [scikit-learn, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/scikit-learn-deep-dive.html)
